In [10]:
import pandas as pd

In [11]:
df_sales = pd.DataFrame({
    "OrderID": [101, 102, 103, 104, 105, 106, 107],
    "Customer": ["Alice", "Bob", "Charlie", "David", "Eve", "Frank", "Grace"],
    "Date": [
        "2023-01-10",     # ISO format
        "10/02/2023",     # DD/MM/YYYY
        "March 3rd, 2023",# verbose format
        "2023/15/04",     # invalid (month 15)
        "2023-06-25",     # ISO
        "25-13-2023",     # invalid (month 13)
        "12 Aug 2023"     # text format
    ],
    "Amount": [500, 650, 300, 450, 900, 200, 700]
})

df_sales

,OrderID,Customer,Date,Amount
0,101,Alice,2023-01-10,500
1,102,Bob,10/02/2023,650
2,103,Charlie,"March 3rd, 2023",300
3,104,David,2023/15/04,450
4,105,Eve,2023-06-25,900
5,106,Frank,25-13-2023,200
6,107,Grace,12 Aug 2023,700


In [12]:
df_returns = pd.DataFrame({
    "OrderID": [102, 105],
    "ReturnDate": ["2023-02-20", "2023-07-10"],
    "Reason": ["Damaged", "Wrong Item"]
})

df_returns

,OrderID,ReturnDate,Reason
0,102,2023-02-20,Damaged
1,105,2023-07-10,Wrong Item


### Converting a column to datetime

<pre>2023-01-10
10/02/2023
March 3rd, 2023
2023/15/04       (invalid)
25-13-2023       (invalid)
12 Aug 2023</pre>


In [15]:
df_sales["Date"] = pd.to_datetime(df_sales["Date"], errors="coerce", dayfirst=True)
display(df_sales)

# errors="coerce" → invalid dates become NaN or NaT
# dayfirst=True → handles “10/02/2023” as 10 Feb (not Oct 2nd)

,OrderID,Customer,Date,Amount
0,101,Alice,2023-01-10,500
1,102,Bob,NaT,650
2,103,Charlie,NaT,300
3,104,David,NaT,450
4,105,Eve,2023-06-25,900
5,106,Frank,NaT,200
6,107,Grace,NaT,700


### Check which dates were invalid

In [16]:
df_sales[df_sales["Date"].isna()]

,OrderID,Customer,Date,Amount
1,102,Bob,NaT,650
2,103,Charlie,NaT,300
3,104,David,NaT,450
5,106,Frank,NaT,200
6,107,Grace,NaT,700


### Extracting components

In [19]:
df_sales["Year"] = df_sales["Date"].dt.year
df_sales["Month"] = df_sales["Date"].dt.month
df_sales["Day"] = df_sales["Date"].dt.day
df_sales["Weekday"] = df_sales["Date"].dt.day_name()

display(df_sales)


,OrderID,Customer,Date,Amount,Year,Month,Day,Weekday
0,101,Alice,2023-01-10,500,2023.0,1.0,10.0,Tuesday
1,102,Bob,NaT,650,NaN,NaN,NaN,NaN
2,103,Charlie,NaT,300,NaN,NaN,NaN,NaN
3,104,David,NaT,450,NaN,NaN,NaN,NaN
4,105,Eve,2023-06-25,900,2023.0,6.0,25.0,Sunday
5,106,Frank,NaT,200,NaN,NaN,NaN,NaN
6,107,Grace,NaT,700,NaN,NaN,NaN,NaN


### Filtering by date

In [ ]:
# Orders after 1st March:
df_sales[df_sales["Date"] > "2023-03-01"]

,OrderID,Customer,Date,Amount,Year,Month,Day,Weekday
4,105,Eve,2023-06-25,900,2023.0,6.0,25.0,Sunday


In [21]:
# Orders in June:
df_sales[df_sales["Month"] == 6]

,OrderID,Customer,Date,Amount,Year,Month,Day,Weekday
4,105,Eve,2023-06-25,900,2023.0,6.0,25.0,Sunday


In [ ]:
# Orders on weekends only
df_sales[df_sales["Weekday"].isin(["Saturday", "Sunday"])]

,OrderID,Customer,Date,Amount,Year,Month,Day,Weekday
4,105,Eve,2023-06-25,900,2023.0,6.0,25.0,Sunday


### Sorting by date

In [24]:
df_sales.sort_values("Date")

,OrderID,Customer,Date,Amount,Year,Month,Day,Weekday
0,101,Alice,2023-01-10,500,2023.0,1.0,10.0,Tuesday
4,105,Eve,2023-06-25,900,2023.0,6.0,25.0,Sunday
1,102,Bob,NaT,650,NaN,NaN,NaN,NaN
2,103,Charlie,NaT,300,NaN,NaN,NaN,NaN
3,104,David,NaT,450,NaN,NaN,NaN,NaN
5,106,Frank,NaT,200,NaN,NaN,NaN,NaN
6,107,Grace,NaT,700,NaN,NaN,NaN,NaN


### Removing / fixing bad dates

##### Remove invalid dates

In [26]:
df_sales_clean = df_sales.dropna(subset=["Date"])
df_sales_clean

,OrderID,Customer,Date,Amount,Year,Month,Day,Weekday
0,101,Alice,2023-01-10,500,2023.0,1.0,10.0,Tuesday
4,105,Eve,2023-06-25,900,2023.0,6.0,25.0,Sunday


##### Fill invalid dates with a default:

In [31]:
df_sales["Date"] = df_sales["Date"].fillna(pd.Timestamp("2023-01-01"))
df_sales

,OrderID,Customer,Date,Amount,Year,Month,Day,Weekday
0,101,Alice,2023-01-10,500,2023.0,1.0,10.0,Tuesday
1,102,Bob,2023-01-01,650,NaN,NaN,NaN,NaN
2,103,Charlie,2023-01-01,300,NaN,NaN,NaN,NaN
3,104,David,2023-01-01,450,NaN,NaN,NaN,NaN
4,105,Eve,2023-06-25,900,2023.0,6.0,25.0,Sunday
5,106,Frank,2023-01-01,200,NaN,NaN,NaN,NaN
6,107,Grace,2023-01-01,700,NaN,NaN,NaN,NaN


### Date arithmetic (Timedelta)

In [ ]:
# Days since the order:
df_sales["DaysAgo"] = (pd.Timestamp("today") - df_sales["Date"]).dt.days
display(df_sales)

,OrderID,Customer,Date,Amount,Year,Month,Day,Weekday,DaysAgo
0,101,Alice,2023-01-10,500,2023.0,1.0,10.0,Tuesday,1058
1,102,Bob,2023-01-01,650,NaN,NaN,NaN,NaN,1067
2,103,Charlie,2023-01-01,300,NaN,NaN,NaN,NaN,1067
3,104,David,2023-01-01,450,NaN,NaN,NaN,NaN,1067
4,105,Eve,2023-06-25,900,2023.0,6.0,25.0,Sunday,892
5,106,Frank,2023-01-01,200,NaN,NaN,NaN,NaN,1067
6,107,Grace,2023-01-01,700,NaN,NaN,NaN,NaN,1067


In [36]:
# Difference between sale date & return date (need merge)
df_merged = pd.merge(df_sales, df_returns, on="OrderID", how="left")
df_merged["ReturnDate"] = pd.to_datetime(df_merged["ReturnDate"])
df_merged["DaysToReturn"] = (df_merged["ReturnDate"] - df_merged["Date"]).dt.days

df_merged

,OrderID,Customer,Date,Amount,Year,Month,Day,Weekday,DaysAgo,ReturnDate,Reason,DaysToReturn
0,101,Alice,2023-01-10,500,2023.0,1.0,10.0,Tuesday,1058,NaT,NaN,NaN
1,102,Bob,2023-01-01,650,NaN,NaN,NaN,NaN,1067,2023-02-20,Damaged,50.0
2,103,Charlie,2023-01-01,300,NaN,NaN,NaN,NaN,1067,NaT,NaN,NaN
3,104,David,2023-01-01,450,NaN,NaN,NaN,NaN,1067,NaT,NaN,NaN
4,105,Eve,2023-06-25,900,2023.0,6.0,25.0,Sunday,892,2023-07-10,Wrong Item,15.0
5,106,Frank,2023-01-01,200,NaN,NaN,NaN,NaN,1067,NaT,NaN,NaN
6,107,Grace,2023-01-01,700,NaN,NaN,NaN,NaN,1067,NaT,NaN,NaN


### Grouping by date components

In [37]:
# Total amount per month:
df_sales.groupby(df_sales["Date"].dt.month)["Amount"].sum()

Date
1    2800
6     900
Name: Amount, dtype: int64

In [38]:
# Total per weekday:
df_sales.groupby(df_sales["Date"].dt.day_name())["Amount"].mean()

Date
Sunday     533.333333
Tuesday    500.000000
Name: Amount, dtype: float64

### Setting datetime as index (TimeSeries Index)

In [40]:
df_ts = df_sales.set_index("Date")
df_ts

,OrderID,Customer,Amount,Year,Month,Day,Weekday,DaysAgo
Date,,,,,,,,
2023-01-10,101,Alice,500,2023.0,1.0,10.0,Tuesday,1058
2023-01-01,102,Bob,650,NaN,NaN,NaN,NaN,1067
2023-01-01,103,Charlie,300,NaN,NaN,NaN,NaN,1067
2023-01-01,104,David,450,NaN,NaN,NaN,NaN,1067
2023-06-25,105,Eve,900,2023.0,6.0,25.0,Sunday,892
2023-01-01,106,Frank,200,NaN,NaN,NaN,NaN,1067
2023-01-01,107,Grace,700,NaN,NaN,NaN,NaN,1067


In [ ]:
# Select orders between two dates:
df_ts.sort_index(inplace=True)  # Need to sort otherwise will throw an error
df_ts["2023-02-01" : "2023-07-01"]

,OrderID,Customer,Amount,Year,Month,Day,Weekday,DaysAgo
Date,,,,,,,,
2023-06-25,105,Eve,900,2023.0,6.0,25.0,Sunday,892


In [ ]:
# Resample weekly:

# Resample() -> regroup the data by time periods
df_ts["Amount"].resample("W").sum()

# "W" = weekly frequency
# Monday → Sunday (default week)

# Output: Groups Amount values week-by-week, then calculates the total Amount for each week.

Date
2023-01-01    2300
2023-01-08       0
2023-01-15     500
2023-01-22       0
2023-01-29       0
2023-02-05       0
2023-02-12       0
2023-02-19       0
2023-02-26       0
2023-03-05       0
2023-03-12       0
2023-03-19       0
2023-03-26       0
2023-04-02       0
2023-04-09       0
2023-04-16       0
2023-04-23       0
2023-04-30       0
2023-05-07       0
2023-05-14       0
2023-05-21       0
2023-05-28       0
2023-06-04       0
2023-06-11       0
2023-06-18       0
2023-06-25     900
Freq: W-SUN, Name: Amount, dtype: int64